# 02 — Features: construcción de Base 2 (`features_df`)

**HealthSignal LATAM — RISA_DATA_V1.0**

## ¿Qué hace este notebook, en palabras simples?

Este notebook toma `timeline_df` (Base 1, la salida de `01_integracion_timeline.ipynb`) y la
convierte en `features_df` — **Base 2**: una tabla con **una fila por paciente y por bloque
de 6 horas** de su episodio de monitoreo, con columnas numéricas resumiendo qué
pasó en ese bloque. Esta es la ÚNICA tabla que después alimenta al motor de reglas y al
Isolation Forest — por eso tiene que quedar bien armada.

**La idea en un dibujo:**

```
Base 1 (timeline_df)                         Base 2 (features_df)
muchas filas "largas",                       una fila "ancha" por
una por cada lectura                         paciente + ventana de 6h
                                  ┌─ agrupar por ventana
  PAT-0001  09:00  HR   74.3     │  y por variable
  PAT-0001  09:20  HR   76.1     │
  PAT-0001  09:05  W_HR 79.0     ├─►  PAT-0001_w1: HR_mean=75.2, HR_zscore=..,
  PAT-0001  09:00  SIG  0.93     │                W_HR_mean=79.0, med_active=0, ...
  PAT-0001  08:30  GLU  145.0    │
  ...                            └─ + contexto (medicación, sueño, comorbilidad...)
```

**Pasos que sigue este notebook:**
1. Cargar Base 1 + las tablas de contexto (`patients`, `encounters`, `healthcare_facilities`,
   `conditions`, `medication_administrations`, `connectivity_events`, `patient_context`,
   `variable_catalog`).
2. Construir la tabla de ventanas fijas de 6h por paciente.
3. Asignar cada fila de Base 1 a su ventana.
4. Agregar (promedio, conteo, cobertura) por paciente + ventana + variable.
5. Calcular z-score y tendencia **personales y causales** (sin mirar el futuro del paciente).
6. Pivotear a formato ancho.
7. Pegar contexto estático (paciente, comorbilidad, centro de salud).
8. Pegar contexto dinámico por solape de fechas (medicación activa, sueño/actividad, conectividad).
9. Validar que no haya fuga de información (leakage).
10. Guardar `../data/processed/features_df.csv`.

## Antes de correr esto

Corre primero `01_integracion_timeline.ipynb` para tener `data/processed/timeline_df.csv`.
Después corre este notebook en orden (`Kernel → Restart & Run All`). La lógica de cada paso
ya se validó contra datos reales (`device_observations`, `laboratory_results` completos, y
las 9 tablas de contexto completas) y contra casos de prueba diseñados para activar cada
regla — está lista para correr con `timeline_df.csv` completo tal cual.


## Paso 0 — Preparar el entorno

In [32]:
import pandas as pd
import numpy as np
import re

DATA_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
WINDOW_HOURS = 6  # tamaño de ventana fijo -- ver justificación en el README

## Paso 1 — Cargar Base 1 y las tablas de contexto

Cargamos `timeline_df.csv` (la salida del notebook 01) y las tablas que vamos a necesitar
como contexto: `patients`, `encounters`, `healthcare_facilities`, `conditions`,
`medication_administrations`, `connectivity_events`, `patient_context` y `variable_catalog`
(para saber cada cuánto se espera una lectura de cada variable, y así calcular cobertura).

Recordatorio de por qué estas tablas NO se apilaron en Base 1: todas tienen un **rango**
(`start_datetime`/`end_datetime`), no un instante — se usan como contexto que se pega por
paciente o por solape de fechas, no como más filas de lecturas.

In [ ]:
timeline_df = pd.read_csv(f"{PROCESSED_DIR}/timeline_df.csv")
timeline_df["timestamp"] = pd.to_datetime(timeline_df["timestamp"], format="mixed")
timeline_df["available_datetime"] = pd.to_datetime(timeline_df["available_datetime"], format="mixed")

patients = pd.read_csv(f"{DATA_DIR}/01_master/patients.csv")
encounters = pd.read_csv(f"{DATA_DIR}/01_master/encounters.csv")
for c in ["start_datetime", "end_datetime"]:
    encounters[c] = pd.to_datetime(encounters[c], format="mixed")
healthcare_facilities = pd.read_csv(f"{DATA_DIR}/01_master/healthcare_facilities.csv")
conditions = pd.read_csv(f"{DATA_DIR}/02_clinical/conditions.csv")
medication_administrations = pd.read_csv(f"{DATA_DIR}/02_clinical/medication_administrations.csv")
for c in ["start_datetime", "end_datetime"]:
    medication_administrations[c] = pd.to_datetime(medication_administrations[c], format="mixed")
connectivity_events = pd.read_csv(f"{DATA_DIR}/04_context/connectivity_events.csv")
for c in ["start_datetime", "end_datetime"]:
    connectivity_events[c] = pd.to_datetime(connectivity_events[c], format="mixed")
patient_context = pd.read_csv(f"{DATA_DIR}/04_context/patient_context.csv")
for c in ["start_datetime", "end_datetime"]:
    patient_context[c] = pd.to_datetime(patient_context[c], format="mixed")
variable_catalog = pd.read_csv(f"{DATA_DIR}/05_metadata/variable_catalog.csv")

# --- CORRECCIÓN (2026-08-27, parte 3): se necesita laboratory_results para el rango de
# referencia clínico de los labs (ver celda de zscore más abajo).
laboratory_results = pd.read_csv(f"{DATA_DIR}/02_clinical/laboratory_results.csv")
# el rango de referencia es constante por test_code en este dataset (confirmado: 1 solo valor
# de reference_low/reference_high por cada uno de LAB_A..D) -- basta con quedarnos con el
# primero de cada uno, no hace falta traerlo lectura por lectura.
rango_referencia_labs = laboratory_results.groupby("test_code")[["reference_low", "reference_high"]].first()

print("timeline_df:", len(timeline_df), "filas,", timeline_df["patient_id"].nunique(), "pacientes")
print("encounters:", len(encounters), "| patients:", len(patients), "| conditions:", len(conditions))
print("medication_administrations:", len(medication_administrations),
      "| connectivity_events:", len(connectivity_events), "| patient_context:", len(patient_context))

## Paso 2 — Construir la tabla de ventanas

Por cada paciente, se recorre su episodio completo (`encounters.start_datetime` a
`end_datetime`) en bloques fijos de **6 horas**, sin solaparse (ventana fija,
"tumbling"). La última ventana de cada paciente puede quedar más corta que
6h si el episodio no es múltiplo exacto — se registra su duración real, no se
descarta ni se rellena.

**¿Por qué 6h y no otro número?** Tiene que ser lo bastante grande para juntar
varios puntos por ventana incluso en la variable con muestreo más espaciado (TEMP, cada
30–240 min según `variable_catalog`) y lo bastante chico para no diluir un cambio agudo en
un promedio de varios días. Con el episodio más corto del dataset (~54h, según el EDA), una
ventana de 6h todavía da varias ventanas por paciente para poder calcular
tendencia.

**Esta tabla no viene de ningún CSV de RISA — la generamos nosotros.** `window_id` es solo
una etiqueta (`f"{patient_id}_w{n}"`) para poder identificar cada bloque.

In [ ]:
def construir_ventanas(encounters_df, horas=WINDOW_HOURS):
    filas = []
    for _, enc in encounters_df.iterrows():
        cursor = enc["start_datetime"]
        fin_episodio = enc["end_datetime"]
        i = 0
        while cursor < fin_episodio:
            cierre = min(cursor + pd.Timedelta(hours=horas), fin_episodio)
            i += 1
            filas.append({
                "patient_id": enc["patient_id"],
                "window_id": f"{enc['patient_id']}_w{i}",
                "window_start": cursor,
                "window_end": cierre,
                "window_duration_h": (cierre - cursor).total_seconds() / 3600,
            })
            cursor = cierre
    return pd.DataFrame(filas)

ventanas = construir_ventanas(encounters)
print("Ventanas generadas:", len(ventanas), "para", ventanas["patient_id"].nunique(), "pacientes")
print("Ventanas por paciente -- min/mediana/max:",
      ventanas.groupby("patient_id").size().agg(["min", "median", "max"]).to_dict())
ventanas.head()

## Paso 3 — Asignar cada fila de Base 1 a su ventana

**Importante (anti-leakage):** para decidir a qué ventana pertenece una fila se usa
`available_datetime`, **no** `timestamp`. Un resultado de laboratorio tomado dentro de la
ventana pero cuyo resultado llega después de que la ventana cerró no puede alimentar esa
ventana — en ese momento el sistema todavía no lo sabía. Para vitales/wearable/dispositivo
esto no cambia nada (`available_datetime == timestamp`), pero para laboratorio sí es
determinante.

**Cuidado con esta celda a escala real.** La forma "obvia" de resolver esto sería cruzar
`timeline_df` completo contra `ventanas` por `patient_id` y después filtrar por fecha — pero
eso arma, para cada paciente, el producto cartesiano de TODAS sus lecturas × TODAS sus
ventanas *antes* de filtrar. Con datos de prueba chicos no se nota, pero con el dataset
completo (~2.5 millones de filas de Base 1 × ~30-100 ventanas por paciente) el cruce
intermedio puede llegar a cientos de millones de filas y tirar un `MemoryError`. La solución:
en vez de cruzar contra TODAS las ventanas, calculamos directamente con aritmética en qué
número de ventana cae cada fila (cuántos bloques de 6h pasaron desde el inicio
del encounter de ese paciente), y solo al final pegamos `window_start`/`window_end` con una
clave exacta (`patient_id` + `window_id`), no una que dispare un cruce masivo.

In [ ]:
# 1) Pegar el inicio/fin del encounter de cada paciente -- esto es 1 fila por paciente,
#    no arma ningún producto cartesiano.
enc_bounds = encounters[["patient_id", "start_datetime", "end_datetime"]].rename(
    columns={"start_datetime": "encounter_start", "end_datetime": "encounter_end"})
con_encounter = timeline_df.merge(enc_bounds, on="patient_id", how="inner")

# 2) Quedarnos solo con lo que cae dentro del rango real del encounter de ese paciente.
dentro_del_encounter = (
    (con_encounter["available_datetime"] >= con_encounter["encounter_start"]) &
    (con_encounter["available_datetime"] < con_encounter["encounter_end"])
)
con_encounter = con_encounter[dentro_del_encounter].copy()

# 3) Calcular en qué ventana cae cada fila con aritmética directa (sin cruzar contra `ventanas`).
horas_transcurridas = (con_encounter["available_datetime"] - con_encounter["encounter_start"]).dt.total_seconds() / 3600
indice_ventana = (horas_transcurridas // WINDOW_HOURS).astype(int) + 1
con_encounter["window_id"] = con_encounter["patient_id"] + "_w" + indice_ventana.astype(str)

# 4) Recién ahora pegamos window_start/window_end/window_duration_h -- por (patient_id,
#    window_id), una clave que matchea 1 a 1 con `ventanas`, no un cruce masivo por patient_id solo.
asignado = con_encounter.merge(
    ventanas[["patient_id", "window_id", "window_start", "window_end", "window_duration_h"]],
    on=["patient_id", "window_id"], how="inner"
)

print(f"Filas de Base 1 asignadas a alguna ventana: {len(asignado)} de {len(timeline_df)}")
print(f"(las que no calzan son lecturas fuera del rango de su encounter -- revisar si el número es alto)")

## Paso 4 — Agregar por paciente + ventana + variable, y calcular cobertura

Se agrupa por `variable_code` (recordatorio: sin esto estaríamos promediando HR con
GLUCOSA en el mismo número, algo sin sentido) y se calcula el promedio y el conteo de
lecturas dentro de la ventana.

La **cobertura** compara cuántas lecturas hubo realmente contra cuántas se esperarían según
`variable_catalog.nominal_sampling` (ej. "5-15 min" para HR → se espera una lectura cada
~10 min). Sirve para que Base 2 sepa distinguir "esta ventana tiene pocos datos, ojo" de
"esta ventana está completa".

In [ ]:
agg = asignado.groupby(["patient_id", "window_id", "variable_code"]).agg(
    valor_medio=("value", "mean"),
    n_lecturas=("value", "count"),
).reset_index()
agg = agg.merge(
    ventanas[["patient_id", "window_id", "window_start", "window_end", "window_duration_h"]],
    on=["patient_id", "window_id"], how="left"
)

def rango_nominal_min(nominal_sampling):
    # "5-15 min" -> (5.0, 15.0) ; variables no periódicas (EVENT, PERIODIC, INTERVAL, EPISODE) -> (NaN, NaN)
    m = re.match(r"(\d+)\s*-\s*(\d+)\s*min", str(nominal_sampling))
    if not m:
        return np.nan, np.nan
    return float(m.group(1)), float(m.group(2))

rango = variable_catalog.set_index("variable_code")["nominal_sampling"].map(rango_nominal_min)
nominal_min = rango.map(lambda t: t[0])
nominal_max = rango.map(lambda t: t[1])

# --- CORRECCIÓN (2026-08-27): el intervalo esperado se calcula EMPÍRICAMENTE a partir del
# dato real (mediana del tiempo entre lecturas consecutivas por variable, en todo el cohorte),
# no del punto medio del rango "nominal_sampling" del catálogo. Ese rango es la promesa de
# diseño del dispositivo, y en RISA la cadencia real observada difiere bastante de esa promesa
# (ej. HR real ~20 min vs. 10 min nominal; WEARABLE_HR/STEPS real ~30 min vs. 10 min nominal;
# TEMP real ~60 min vs. 135 min nominal; SBP/DBP real ~120 min vs. 195 min nominal). Con el
# punto medio nominal, 6 de las 8 variables periódicas quedaban con coverage saturado en 1.0
# todo el tiempo o pegado muy por debajo de 1.0 sin nunca acercarse -- una columna casi
# constante que no refleja cobertura real y no aporta señal al modelo.
lecturas_variable = asignado[["patient_id", "variable_code", "available_datetime"]].dropna()
lecturas_variable = lecturas_variable.sort_values(["patient_id", "variable_code", "available_datetime"])
gap_min = lecturas_variable.groupby(["patient_id", "variable_code"])["available_datetime"].diff().dt.total_seconds() / 60
intervalo_empirico_min = gap_min.groupby(lecturas_variable["variable_code"]).median()

# Solo se usa el intervalo empírico para variables que SÍ son periódicas según el catálogo
# (nominal_min no es NaN); para EVENT/PERIODIC/INTERVAL/EPISODE se mantiene NaN a propósito
# -- no se puede calcular "cobertura" de algo que no es periódico.
intervalo_esperado_min = intervalo_empirico_min.reindex(nominal_min.index)
intervalo_esperado_min = intervalo_esperado_min.where(nominal_min.notna())

# Salvavidas de sanity: si el intervalo empírico se sale mucho del rango declarado por el
# catálogo (dato ruidoso / pocas lecturas para estimarlo bien), se acota entre la mitad del
# mínimo nominal y el doble del máximo nominal -- así un outlier no dispara un intervalo
# absurdo, pero sigue mandando el dato real observado, no el spec de diseño del dispositivo.
intervalo_esperado_min = intervalo_esperado_min.clip(lower=nominal_min * 0.5, upper=nominal_max * 2.0)

agg["intervalo_esperado_min"] = agg["variable_code"].map(intervalo_esperado_min)
agg["n_esperado"] = (agg["window_duration_h"] * 60) / agg["intervalo_esperado_min"]
agg["coverage"] = (agg["n_lecturas"] / agg["n_esperado"]).clip(upper=1.0)
# coverage máximo teórico = 1.0 (100%) -- nunca debe superarlo, es una proporción lecturas
# reales / lecturas esperadas, y aquí queda acotada explícitamente con el clip.

print("Intervalo esperado por variable (empírico, minutos, acotado por el catálogo):")
print(intervalo_esperado_min.dropna().sort_index())
print()
print(agg.head(8)[["patient_id", "window_id", "variable_code", "valor_medio", "n_lecturas", "coverage"]])


## Paso 5 — Z-score y tendencia personales, calculados de forma causal

El z-score compara el valor de esta ventana contra el **baseline propio del paciente**
(no contra los otros 999) — pero solo usando ventanas **anteriores**, nunca la ventana
actual ni ventanas futuras. Si se incluyera la ventana actual en su propio baseline, un
cambio real quedaría diluido (se compararía contra sí mismo) y además sería una fuga de
información hacia el futuro dentro del propio dato.

Por eso se usa `.shift(1)` (ignora la fila actual) seguido de `.expanding()` (promedio de
todo lo anterior) dentro de cada grupo paciente+variable, ordenado por tiempo. La
**primera ventana de cada paciente para cada variable no tiene baseline todavía** — su
z-score sale `NaN`, y es lo correcto (no hay con qué comparar, "arranque en frío").

In [ ]:
Z_CAP = 6.0                     # winsorización -- +/-6 desviaciones (o +/-6 anchos de rango de referencia
                                # para labs) ya es clínicamente extremo, no hace falta más rango.
MIN_HISTORIAL_PERIODICAS = 3   # HR, RR, SpO2, TEMP, SBP, DBP, WEARABLE_HR, STEPS, SIGNAL_QUALITY_INDEX, ACTIVITY_LEVEL: se
                                # muestrean cada 5-360 min, cualquier paciente acumula de sobra 3 lecturas previas.

tipo_muestreo = variable_catalog.set_index("variable_code")["nominal_sampling"]
es_evento = agg["variable_code"].map(tipo_muestreo) == "EVENT"   # LAB_A..D

agg = agg.sort_values(["patient_id", "variable_code", "window_start"])
grp = agg.groupby(["patient_id", "variable_code"])["valor_medio"]

agg["baseline_mean"] = grp.transform(lambda s: s.shift(1).expanding().mean())
agg["baseline_std"] = grp.transform(lambda s: s.shift(1).expanding().std())
agg["baseline_n"] = grp.transform(lambda s: s.shift(1).expanding().count())
agg["tendencia"] = grp.transform(lambda s: s.diff())  # cambio vs. la ventana inmediatamente anterior

# --- CORRECCIÓN (2026-08-27, parte 1): con baseline_std=0 (historial reciente idéntico -- muy
# común en STEPS y SIGNAL_QUALITY_INDEX, ej. varias ventanas seguidas en 0 pasos) el cociente
# daba ±inf (confirmado: 798 filas en STEPS_zscore, 28 en SIGNAL_QUALITY_INDEX_zscore). Y con
# muy poco historial (2-3 puntos) el std es tan inestable que el z-score "finito" igual se
# disparaba a cientos/miles de desviaciones (ej. HR_zscore llegaba a -2329, DBP_zscore a
# +711) -- ninguno de los dos casos es algo que Isolation Forest pueda recibir: no acepta NaN
# ni inf, y una sola columna con valores en los miles domina todos los splits del bosque.
zscore_personal = (agg["valor_medio"] - agg["baseline_mean"]) / agg["baseline_std"]
zscore_personal = zscore_personal.replace([np.inf, -np.inf], np.nan)

agg["baseline_insuficiente"] = agg["baseline_n"] < MIN_HISTORIAL_PERIODICAS
# sin historial suficiente para confiar en el baseline -> NaN explícito (se marca con el flag,
# no se inventa un z-score con menos lecturas de referencia de las que la variable necesita)
zscore_personal = zscore_personal.where(~agg["baseline_insuficiente"])

# --- CORRECCIÓN (2026-08-27, parte 3): para variables tipo EVENT (LAB_A..D) el z-score
# personal es matemáticamente inalcanzable para casi todo el mundo -- la mediana real es 1
# sola lectura por paciente en TODO su encounter, y un std necesita al menos 2 lecturas
# previas para existir (con 1 sola, pandas devuelve NaN: no hay variación que medir con un
# solo punto). Ya se probó bajar el mínimo de historial a 1 y no cambió nada (seguían
# saliendo ~43-44 valores no nulos de 24,859), porque el bloqueo real no era el umbral, era
# la matemática del std. En su lugar, para labs se usa una desviación respecto al RANGO DE
# REFERENCIA clínico (reference_low/reference_high de laboratory_results, constante por
# test_code) -- esa sí existe desde la primera lectura, no depende del historial del
# paciente, y es clínicamente más razonable para un valor que se mide una sola vez: 0 si el
# resultado cae dentro del rango normal, y una fracción del ancho del rango (negativa por
# debajo, positiva por encima) si cae fuera.
ref = agg["variable_code"].map(rango_referencia_labs["reference_low"]).rename("reference_low").to_frame()
ref["reference_high"] = agg["variable_code"].map(rango_referencia_labs["reference_high"])
ancho_rango = ref["reference_high"] - ref["reference_low"]

desviacion_ref = pd.Series(0.0, index=agg.index)
bajo_rango = agg["valor_medio"] < ref["reference_low"]
alto_rango = agg["valor_medio"] > ref["reference_high"]
desviacion_ref[bajo_rango] = (agg.loc[bajo_rango, "valor_medio"] - ref.loc[bajo_rango, "reference_low"]) / ancho_rango[bajo_rango]
desviacion_ref[alto_rango] = (agg.loc[alto_rango, "valor_medio"] - ref.loc[alto_rango, "reference_high"]) / ancho_rango[alto_rango]
# sin rango de referencia válido (variable_code sin match en el catálogo de labs, o ancho <= 0) -> NaN, no se inventa
desviacion_ref = desviacion_ref.where(ancho_rango.notna() & (ancho_rango > 0) & agg["valor_medio"].notna())

zscore_final = zscore_personal.where(~es_evento, desviacion_ref)
agg["zscore"] = zscore_final.clip(-Z_CAP, Z_CAP)
# para labs, "baseline_insuficiente" ya no aplica (no usan baseline personal) -- se deja en False
agg.loc[es_evento, "baseline_insuficiente"] = False

ejemplo = agg[agg["variable_code"] == "HR"].sort_values(["patient_id", "window_start"])
print("Ejemplo (primeras filas con HR): la 1a ventana de cada paciente sale con zscore NaN, es esperado.")
print(ejemplo[["patient_id", "window_id", "valor_medio", "baseline_mean", "baseline_n", "zscore", "tendencia"]].head(8))
print()

ejemplo_lab = agg[(agg["variable_code"] == "LAB_A") & agg["zscore"].notna()].sort_values(["patient_id", "window_start"])
print("Ejemplo (LAB_A con desviación de rango de referencia calculada):")
print(ejemplo_lab[["patient_id", "window_id", "valor_medio", "zscore"]].head(8))
print()

n_inf_antes = int(np.isinf((agg["valor_medio"] - agg["baseline_mean"]) / agg["baseline_std"]).sum())
print(f"±inf que había antes de la corrección: {n_inf_antes}")
print(f"z-score winsorizado a [-{Z_CAP}, {Z_CAP}].")
print()
print("Cobertura de zscore no-nulo por variable (antes vs. ahora debería subir fuerte en LAB_A..D):")
print(agg.groupby("variable_code")["zscore"].apply(lambda s: f"{s.notna().sum()} de {len(s)} ({s.notna().mean()*100:.1f}%)"))


## Paso 6 — Pivotear a formato ancho

Hasta acá `agg` tiene varias filas por ventana (una por variable). Pivotear pone cada
variable×estadístico como su propia columna, para terminar con **una sola fila por
paciente+ventana** — el formato que necesita el modelo.

Un detalle técnico importante: si alguna combinación variable×estadístico sale
completamente vacía en todo el dataset (por ejemplo, muy poca historia todavía para calcular
tendencia de una variable rara), `pivot_table` puede "perderla" en vez de dejarla vacía. Para
que el esquema de columnas de `features_df` sea siempre el mismo pase lo que pase, se
completan explícitamente todas las columnas esperadas.

In [ ]:
pivot = agg.pivot_table(
    index=["patient_id", "window_id"],
    columns="variable_code",
    values=["valor_medio", "n_lecturas", "coverage", "zscore", "tendencia", "baseline_insuficiente"],
)
pivot.columns = [f"{var}_{stat}" for stat, var in pivot.columns]
features_df = pivot.reset_index()

todas_las_variables = sorted(agg["variable_code"].unique())
todos_los_estadisticos = ["valor_medio", "n_lecturas", "coverage", "zscore", "tendencia", "baseline_insuficiente"]
for var in todas_las_variables:
    for stat in todos_los_estadisticos:
        col = f"{var}_{stat}"
        if col not in features_df.columns:
            features_df[col] = np.nan

# Nos aseguramos de que TODAS las ventanas generadas en el Paso 2 aparezcan en features_df,
# aunque no hayan tenido ninguna lectura en Base 1 (esa ventana queda con todo NaN -- es
# información real: "no hubo datos en este bloque", no se descarta la ventana)
features_df = ventanas.merge(features_df, on=["patient_id", "window_id"], how="left")

print("features_df tras pivotear:", features_df.shape)
[c for c in features_df.columns if c.startswith("HR_")]

## Paso 7 — Pegar el contexto ESTÁTICO (mismo valor en todas las ventanas del paciente)

`patients` (edad, sexo, programa de cuidado, perfil de riesgo basal), `encounters`
(centro/facility, tipo de cuidado), `healthcare_facilities` (madurez digital del centro) y
`conditions` (comorbilidad, convertida de filas a columnas binarias `tiene_<categoría>`) se
pegan por `patient_id` — el mismo valor se repite en todas las ventanas de ese paciente,
porque son atributos que no cambian de una ventana a otra.

In [ ]:
enc_slim = encounters[["patient_id", "facility_id", "care_setting", "encounter_type"]].drop_duplicates("patient_id")
patients_slim = patients[["patient_id", "sex_at_birth", "age_years", "age_group", "care_program", "baseline_risk_profile"]]

comorbilidad = pd.crosstab(conditions["patient_id"], conditions["condition_category"])
comorbilidad = (comorbilidad > 0).astype(int)
comorbilidad.columns = [f"tiene_{c.lower()}" for c in comorbilidad.columns]
comorbilidad = comorbilidad.reset_index()

features_df = (features_df
               .merge(patients_slim, on="patient_id", how="left")
               .merge(enc_slim, on="patient_id", how="left")
               .merge(healthcare_facilities[["facility_id", "digital_maturity", "connectivity_profile"]],
                      on="facility_id", how="left")
               .merge(comorbilidad, on="patient_id", how="left"))

for c in comorbilidad.columns:
    if c != "patient_id":
        features_df[c] = features_df[c].fillna(0).astype(int)

print("features_df tras contexto estático:", features_df.shape)

## Paso 8 — Pegar el contexto DINÁMICO por solape de fechas

`medication_administrations`, `connectivity_events` y `patient_context` tienen un rango
propio (`start_datetime`/`end_datetime`) que puede o no cruzarse con una ventana dada. La
función `unir_por_solape` cruza cada ventana con los intervalos del MISMO paciente y se
queda solo con los que realmente se solapan en el tiempo (no hace falta que el intervalo
quepa completo dentro de la ventana, con que se toquen alcanza).

- **Medicación activa**: si algún tratamiento se solapa con la ventana, `med_active = 1`.
- **Conectividad**: si hay más de un evento solapado en la misma ventana, se conserva el
  peor estado (`DISCONNECTED` > `INTERMITTENT` > `DELAYED_SYNC`).
- **`patient_context`**: puede haber más de un registro contradictorio para el mismo tipo de
  contexto en el mismo momento (el EDA encontró pares contradictorios y solapes) — se resuelve
  quedándose con el de mayor `confidence`. Después se pivotea `context_type`
  (`PHYSICAL_ACTIVITY`, `RECOVERY_PHASE`, `SLEEP_STATE`) a columnas.

In [ ]:
def unir_por_solape(ventanas_df, intervalos_df, start_col="start_datetime", end_col="end_datetime"):
    m = ventanas_df[["patient_id", "window_id", "window_start", "window_end"]].merge(
        intervalos_df, on="patient_id", how="inner")
    solapa = (m["window_start"] < m[end_col]) & (m[start_col] < m["window_end"])
    return m[solapa]

# -- medicación activa --
solape_med = unir_por_solape(ventanas, medication_administrations)
med_activa = solape_med.groupby(["patient_id", "window_id"]).size().reset_index(name="_n")
med_activa["med_active"] = 1
features_df = features_df.merge(med_activa[["patient_id", "window_id", "med_active"]],
                                 on=["patient_id", "window_id"], how="left")
features_df["med_active"] = features_df["med_active"].fillna(0).astype(int)

# -- conectividad: el peor estado si hay más de uno solapado --
PRIORIDAD_CONECTIVIDAD = {"DISCONNECTED": 3, "INTERMITTENT": 2, "DELAYED_SYNC": 1}
solape_conn = unir_por_solape(ventanas, connectivity_events)
if len(solape_conn):
    solape_conn = solape_conn.copy()
    solape_conn["_prioridad"] = solape_conn["connectivity_status"].map(PRIORIDAD_CONECTIVIDAD).fillna(0)
    peor = solape_conn.sort_values("_prioridad", ascending=False).drop_duplicates(["patient_id", "window_id"])
    features_df = features_df.merge(
        peor[["patient_id", "window_id", "connectivity_status"]].rename(columns={"connectivity_status": "connectivity_flag"}),
        on=["patient_id", "window_id"], how="left")
else:
    features_df["connectivity_flag"] = np.nan

# -- patient_context: resolver conflictos por confidence, pivotear context_type a columnas --
solape_ctx = unir_por_solape(ventanas, patient_context)
if len(solape_ctx):
    solape_ctx = solape_ctx.sort_values("confidence", ascending=False)
    resuelto = solape_ctx.drop_duplicates(["patient_id", "window_id", "context_type"], keep="first")
    ctx_pivot = resuelto.pivot_table(index=["patient_id", "window_id"], columns="context_type",
                                      values="context_value", aggfunc="first")
    ctx_pivot.columns = [f"context_{c.lower()}" for c in ctx_pivot.columns]
    features_df = features_df.merge(ctx_pivot.reset_index(), on=["patient_id", "window_id"], how="left")

print("features_df tras contexto dinámico:", features_df.shape)
cols_contexto = ["med_active", "connectivity_flag"] + [c for c in features_df.columns if c.startswith("context_")]
features_df[["patient_id", "window_id"] + cols_contexto].head(10)

## Paso 9 — Validar antes de guardar

El único chequeo que no puede fallar: ninguna fila usada en la agregación puede tener un
`available_datetime` posterior al cierre de su ventana (sería estar usando información que
todavía no existía en ese momento).

In [ ]:
assert (asignado["available_datetime"] < asignado["window_end"]).all(), \
    "Hay una fila cuyo available_datetime cae fuera de su ventana -- hay leakage, revisar el Paso 3"
print("Validación anti-leakage: OK")
print(f"\nfeatures_df final: {features_df.shape[0]} filas x {features_df.shape[1]} columnas")
print(f"Pacientes cubiertos: {features_df['patient_id'].nunique()}")

## Paso 10 — Guardar Base 2

Esta es la tabla que va a leer el notebook de detección (reglas + Isolation Forest).

In [ ]:
import os
os.makedirs(PROCESSED_DIR, exist_ok=True)
out_path = f"{PROCESSED_DIR}/features_df.csv"
features_df.to_csv(out_path, index=False)
print(f"Guardado: {out_path}")

## Notas y decisiones a declarar en el README

- **Ventanas fijas de 6h, sin solape** (no deslizantes) — decisión explícita por simplicidad
  y tiempo del hackathon; una mejora futura sería pasar a ventanas deslizantes para no
  partir en dos una tendencia que cruza el borde de una ventana.
- **`coverage` queda `NaN` para variables no periódicas** (labs tipo `EVENT`, `ACTIVITY_LEVEL`
  tipo `INTERVAL`, `SLEEP_STATE` tipo `EPISODE`) — no tiene sentido calcular "cobertura
  esperada" de algo que no ocurre a intervalos regulares. Es una limitación conocida, no un
  bug.
- **z-score y tendencia sale `NaN` en la primera ventana de cada paciente para cada
  variable** ("arranque en frío") — no hay baseline previo con el cual comparar. Documentar
  esto explícitamente en el README como limitación conocida del enfoque.
- **La deduplicación entre registros contradictorios de `patient_context` se resuelve por
  `confidence`** — si dos registros empatan en confidence, pandas se queda con el primero
  según el orden de `sort_values` (estable); no hay una regla de desempate adicional todavía.
- **`device_observations` no se usó en Base 2 como variable de riesgo** (solo aportó
  `SIGNAL_QUALITY_INDEX_*`, ya lo sabíamos desde `01_integracion_timeline`) — sigue
  disponible ahí por si se quiere usar como ponderador de confianza más adelante.


In [ ]:
features_df.head()

In [ ]:
features_df.info()